# Censo Escolar (INEP): municípios das 3 regiões, 2010 x 2022 x 2025

Compara escolas, matrículas, turmas, professores e infraestrutura das escolas nos **21 municípios** dos três recortes deste repositório
(Candelária e vizinhos, Novo Cabrais e vizinhos e a microrregião de Santa Cruz do Sul), com o **Rio Grande do Sul** e o **Brasil** como referência.

Fonte: microdados do Censo Escolar da Educação Básica do INEP (um arquivo por ano, com uma linha por escola).
Cada ano é um zip de 100 a 300 MB; o notebook baixa, filtra e apaga. Com os 3 anos leva de 5 a 15 minutos.

**No fim, o notebook baixa `censo_escolar_regioes.zip`. Mande esse zip de volta** (planilha com os números exatos e os gráficos).

In [ ]:
import os, re, io, time, zipfile, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_rows', 300); pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 10})

# Anos a comparar. Dá para pôr qualquer ano de 2007 a 2025 (cada um é mais um download).
ANOS = [2010, 2022, 2025]

# Código IBGE -> (nome, recortes a que pertence)
CAND, NC, MICRO = 'Candelária e vizinhos', 'Novo Cabrais e vizinhos', 'Microrregião de Santa Cruz do Sul'
MUNICIPIOS = {
    4301206: ('Arroio do Tigre', [MICRO]),
    4303004: ('Cachoeira do Sul', [CAND, NC]),
    4304200: ('Candelária', [CAND, NC, MICRO]),
    4305132: ('Cerro Branco', [CAND, NC]),
    4307815: ('Estrela Velha', [MICRO]),
    4309159: ('Gramado Xavier', [MICRO]),
    4309571: ('Herveiras', [MICRO]),
    4309753: ('Ibarama', [MICRO]),
    4311239: ('Lagoa Bonita do Sul', [CAND, MICRO]),
    4312153: ('Mato Leitão', [MICRO]),
    4313391: ('Novo Cabrais', [CAND, NC]),
    4314027: ('Paraíso do Sul', [NC]),
    4314068: ('Passa Sete', [CAND, MICRO]),
    4315701: ('Rio Pardo', [CAND]),
    4316808: ('Santa Cruz do Sul', [MICRO]),
    4320263: ('Segredo', [MICRO]),
    4320677: ('Sinimbu', [MICRO]),
    4320701: ('Sobradinho', [MICRO]),
    4322533: ('Vale do Sol', [CAND, MICRO]),
    4322608: ('Venâncio Aires', [MICRO]),
    4322707: ('Vera Cruz', [CAND, MICRO]),
}
NOME = {c: n for c, (n, _) in MUNICIPIOS.items()}
MUNIS = sorted(NOME.values())
REFS = ['RS', 'Brasil']
print(len(MUNICIPIOS), 'municípios')

## 1. Baixar e filtrar os microdados

Colunas usadas (uma linha por escola): situação de funcionamento, rede (federal, estadual, municipal, privada), zona (urbana ou rural),
matrículas por etapa, turmas, professores e itens de infraestrutura. Colunas que não existem em algum ano ficam vazias.

In [ ]:
URL = 'https://download.inep.gov.br/dados_abertos/microdados_censo_escolar_{ano}{sufixo}.zip'
ID = ['NU_ANO_CENSO', 'CO_UF', 'CO_MUNICIPIO', 'CO_ENTIDADE', 'TP_DEPENDENCIA', 'TP_LOCALIZACAO', 'TP_SITUACAO_FUNCIONAMENTO']
MAT = {
    'QT_MAT_BAS': 'Educação básica (total)',
    'QT_MAT_INF_CRE': 'Creche', 'QT_MAT_INF_PRE': 'Pré-escola',
    'QT_MAT_FUND_AI': 'Fundamental, anos iniciais', 'QT_MAT_FUND_AF': 'Fundamental, anos finais',
    'QT_MAT_MED': 'Ensino médio', 'QT_MAT_PROF': 'Educação profissional',
    'QT_MAT_EJA': 'EJA (jovens e adultos)', 'QT_MAT_ESP': 'Educação especial',
}
OUTROS_QT = {'QT_TUR_BAS': 'Turmas', 'QT_DOC_BAS': 'Professores (soma por escola)'}
INFRA = {
    'IN_INTERNET': 'Internet', 'IN_BANDA_LARGA': 'Banda larga',
    'IN_BIBLIOTECA_SALA_LEITURA': 'Biblioteca ou sala de leitura', 'IN_LABORATORIO_INFORMATICA': 'Laboratório de informática',
    'IN_LABORATORIO_CIENCIAS': 'Laboratório de ciências', 'IN_QUADRA_ESPORTES': 'Quadra de esportes',
    'IN_ESGOTO_REDE_PUBLICA': 'Esgoto da rede pública', 'IN_AGUA_POTAVEL': 'Água potável',
    'IN_ACESSIBILIDADE_INEXISTENTE': 'Sem nenhum item de acessibilidade', 'IN_ALIMENTACAO': 'Oferece alimentação',
}
COLUNAS = ID + list(MAT) + list(OUTROS_QT) + list(INFRA)

def baixar(ano):
    destino = f'censo_escolar_{ano}.zip'
    if os.path.exists(destino):
        return destino
    for sufixo in ['', '_']:          # o de 2025 se chama ..._2025_.zip
        url = URL.format(ano=ano, sufixo=sufixo)
        for verificar in [True, False]:  # o site do INEP às vezes falha na verificação do certificado
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    r = requests.get(url, stream=True, timeout=120, verify=verificar)
                if r.status_code == 404:
                    break
                r.raise_for_status()
                with open(destino, 'wb') as f:
                    for parte in r.iter_content(1 << 20):
                        f.write(parte)
                print(f'  {ano}: {os.path.getsize(destino) / 1e6:,.0f} MB de {url}')
                return destino
            except requests.exceptions.SSLError:
                continue
            except Exception as e:
                print(f'  {ano}: falhou ({e}); tentando de novo em 10 s'); time.sleep(10)
    raise RuntimeError(f'Não consegui baixar {ano}. Confira o link em gov.br/inep (Dados abertos > Microdados > Censo Escolar).')

def separador(z, nome):
    with z.open(nome) as f:
        linha = f.readline().decode('latin-1')
    return ';' if linha.count(';') >= linha.count(',') else ','

def cabecalho(z, nome):
    with z.open(nome) as f:
        return list(pd.read_csv(f, sep=separador(z, nome), encoding='latin-1', nrows=0).columns)

def ler(ano):
    """Lê o CSV de escolas do ano e devolve: escolas dos 21 municípios e totais por UF (para RS e Brasil)."""
    z = zipfile.ZipFile(baixar(ano))
    csvs = [n for n in z.namelist() if n.lower().endswith('.csv')]
    # até 2024: microdados_ed_basica_AAAA.csv; em 2025: Tabela_Escola_2025_V2.csv (e outras tabelas)
    alvo = [n for n in csvs if re.search(r'(ed_basica|tabela_escola)_?\d{4}(_v\d+)?\.csv$', n.lower())]
    if not alvo:
        alvo = sorted((n for n in csvs if 'CO_MUNICIPIO' in cabecalho(z, n) and 'CO_ENTIDADE' in cabecalho(z, n)),
                      key=lambda n: -z.getinfo(n).file_size)
    if not alvo:
        print('Arquivos no zip:', *csvs, sep='\n  '); raise RuntimeError('CSV de escolas não encontrado')
    cab = cabecalho(z, alvo[0])
    usar = [c for c in COLUNAS if c in cab]
    faltam = [c for c in COLUNAS if c not in cab]
    if faltam:
        print(f'  {ano}: colunas ausentes em {os.path.basename(alvo[0])} (ficam vazias): {", ".join(faltam)}')
    if 'QT_MAT_BAS' not in cab:
        # Mostra as colunas para ajustar o notebook se o INEP mudou o formato
        print(f'\n  ATENÇÃO {ano}: sem QT_MAT_BAS. Mande esta lista de volta:')
        for n in csvs:
            print(f'  {os.path.basename(n)}: {", ".join(cabecalho(z, n))}')
    munis, ufs = [], []
    with z.open(alvo[0]) as f:
        for bloco in pd.read_csv(f, sep=separador(z, alvo[0]), encoding='latin-1', usecols=usar, chunksize=50_000, low_memory=False):
            bloco = bloco.reindex(columns=COLUNAS)
            if bloco['TP_SITUACAO_FUNCIONAMENTO'].isna().all():
                bloco['TP_SITUACAO_FUNCIONAMENTO'] = 1   # tabela que já traz só escolas em atividade
            if bloco['CO_UF'].isna().all():
                bloco['CO_UF'] = bloco['CO_MUNICIPIO'] // 100000
            ativa = bloco[bloco['TP_SITUACAO_FUNCIONAMENTO'] == 1]
            munis.append(ativa[ativa['CO_MUNICIPIO'].isin(MUNICIPIOS)])
            ufs.append(resumir(ativa, 'CO_UF'))
    escolas = pd.concat(munis, ignore_index=True)
    escolas['ano'] = ano
    uf = pd.concat(ufs).groupby(level=0).sum(min_count=1)
    return escolas, uf

def resumir(df, chave):
    """Soma por chave: escolas (total, por rede, rurais), matrículas, turmas, professores, escolas com cada item."""
    d = df.assign(
        escolas=1,
        esc_municipal=(df['TP_DEPENDENCIA'] == 3).astype(int), esc_estadual=(df['TP_DEPENDENCIA'] == 2).astype(int),
        esc_federal=(df['TP_DEPENDENCIA'] == 1).astype(int), esc_privada=(df['TP_DEPENDENCIA'] == 4).astype(int),
        esc_rural=(df['TP_LOCALIZACAO'] == 2).astype(int),
        mat_rural=np.where(df['TP_LOCALIZACAO'] == 2, df['QT_MAT_BAS'], 0),
        mat_publica=np.where(df['TP_DEPENDENCIA'] != 4, df['QT_MAT_BAS'], 0),
        mat_municipal=np.where(df['TP_DEPENDENCIA'] == 3, df['QT_MAT_BAS'], 0),
        mat_estadual=np.where(df['TP_DEPENDENCIA'] == 2, df['QT_MAT_BAS'], 0),
        mat_privada=np.where(df['TP_DEPENDENCIA'] == 4, df['QT_MAT_BAS'], 0),
    )
    # escolas que responderam cada item de infraestrutura (base do %)
    for c in INFRA:
        d[f'base_{c}'] = d[c].notna().astype(int)
    soma = ['escolas', 'esc_municipal', 'esc_estadual', 'esc_federal', 'esc_privada', 'esc_rural',
            'mat_rural', 'mat_publica', 'mat_municipal', 'mat_estadual', 'mat_privada',
            *MAT, *OUTROS_QT, *INFRA, *[f'base_{c}' for c in INFRA]]
    return d.groupby(chave)[soma].sum(min_count=1)

ESCOLAS, TOT = [], {}
for ano in ANOS:
    t = time.time()
    cache = f'cache_{ano}.pkl'
    if os.path.exists(cache):
        esc, uf = pd.read_pickle(cache)
    else:
        esc, uf = ler(ano)
        pd.to_pickle((esc, uf), cache)
    ESCOLAS.append(esc)
    m = resumir(esc, 'CO_MUNICIPIO').rename(index=NOME)
    TOT[ano] = pd.concat([m, uf.loc[[43]].rename(index={43: 'RS'}), uf.sum(min_count=1).to_frame('Brasil').T])
    for c in INFRA:   # item com zero no Brasil inteiro = pergunta que não existia naquele ano
        if TOT[ano].loc['Brasil', c] == 0:
            TOT[ano][c] = np.nan
    if os.path.exists(f'censo_escolar_{ano}.zip'):
        os.remove(f'censo_escolar_{ano}.zip')
    print(f'{ano}: {len(esc):,} escolas em atividade nos 21 municípios; {uf["escolas"].sum():,.0f} no Brasil ({time.time() - t:.0f} s)')
ESCOLAS = pd.concat(ESCOLAS, ignore_index=True)

## 2. Indicadores por município

In [ ]:
def indicadores(s):
    """Converte as somas em indicadores legíveis."""
    out = pd.DataFrame(index=s.index)
    out['Escolas em atividade'] = s['escolas']
    out['Escolas municipais'] = s['esc_municipal']
    out['Escolas estaduais'] = s['esc_estadual']
    out['Escolas federais'] = s['esc_federal']
    out['Escolas privadas'] = s['esc_privada']
    out['Escolas rurais'] = s['esc_rural']
    for c, nome in MAT.items():
        out[f'Matrículas: {nome}'] = s[c]
    out['% das matrículas na rede pública'] = 100 * s['mat_publica'] / s['QT_MAT_BAS']
    out['% das matrículas na rede municipal'] = 100 * s['mat_municipal'] / s['QT_MAT_BAS']
    out['% das matrículas na rede estadual'] = 100 * s['mat_estadual'] / s['QT_MAT_BAS']
    out['% das matrículas na rede privada'] = 100 * s['mat_privada'] / s['QT_MAT_BAS']
    out['% das matrículas em escolas rurais'] = 100 * s['mat_rural'] / s['QT_MAT_BAS']
    out['Turmas'] = s['QT_TUR_BAS']
    out['Alunos por turma'] = s['QT_MAT_BAS'] / s['QT_TUR_BAS']
    out['Professores (soma por escola)'] = s['QT_DOC_BAS']
    out['Alunos por professor'] = s['QT_MAT_BAS'] / s['QT_DOC_BAS']
    for c, nome in INFRA.items():
        out[f'% de escolas com: {nome}' if 'Sem' not in nome else f'% de escolas: {nome}'] = 100 * s[c] / s[f'base_{c}'].replace(0, np.nan)
    return out

IND = {ano: indicadores(TOT[ano]).reindex(MUNIS + REFS) for ano in ANOS}
LONGO = pd.concat({ano: df.stack(future_stack=True) for ano, df in IND.items()}).rename('valor').reset_index()
LONGO.columns = ['ano', 'local', 'indicador', 'valor']
TABELA = LONGO.pivot_table(index=['indicador', 'local'], columns='ano', values='valor', aggfunc='first', dropna=False)
TABELA = TABELA.reindex(pd.MultiIndex.from_product([IND[ANOS[0]].columns, MUNIS + REFS]))
A0 = ANOS[0]
# Último ano com dado de cada indicador (em 2025 o INEP tirou matrículas, turmas e professores da tabela de escolas)
ULT = {i: max([a for a in ANOS if TABELA.loc[i, a].notna().any()], default=A0) for i in IND[A0].columns}
ult = TABELA.index.get_level_values(0).map(ULT)
final = np.array([TABELA.iloc[k][a] for k, a in enumerate(ult)], dtype=float)
TABELA['Último ano com dado'] = ult
TABELA[f'Variação desde {A0}'] = final - TABELA[A0]
TABELA[f'Variação % desde {A0}'] = 100 * (final / TABELA[A0] - 1)
TABELA.loc[TABELA.index.get_level_values(0).str.startswith('%'), f'Variação % desde {A0}'] = np.nan

def mostra(indicador):
    print(f'\n{indicador.upper()}'); print(TABELA.loc[indicador].round(1).to_string())

for i in ['Escolas em atividade', 'Escolas rurais', 'Matrículas: Educação básica (total)', 'Matrículas: Creche',
          'Matrículas: Ensino médio', '% das matrículas na rede municipal', 'Alunos por turma', '% de escolas com: Internet']:
    mostra(i)

## 3. Recortes (soma dos municípios de cada região)

As três regiões se sobrepõem (Candelária está nas três), então os totais não devem ser somados entre si.

In [ ]:
REC = {}
for ano in ANOS:
    linhas = {}
    for rec in [CAND, NC, MICRO]:
        codigos = [c for c, (_, r) in MUNICIPIOS.items() if rec in r]
        linhas[rec] = TOT[ano].loc[[NOME[c] for c in codigos]].sum(min_count=1)
    REC[ano] = indicadores(pd.DataFrame(linhas).T)
RECORTES = pd.concat(REC, names=['ano', 'recorte']).T
print(RECORTES.round(1).to_string())

## 4. Gráficos

In [ ]:
os.makedirs('graficos', exist_ok=True)
COR_A0, COR_A1, COR_REF, COR_VIZ = '#b8b8b8', '#1f5f99', '#555555', '#1f5f99'
def br(x, casas=1):
    if pd.isna(x) or np.isinf(x): return 's/d'
    return f'{x:,.{casas}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

def halteres(indicador, arquivo, sufixo='', casas=1, refs=True):
    A1 = ULT[indicador]
    locais = MUNIS + (['RS'] if refs else [])
    d = TABELA.loc[indicador].loc[locais, [A0, A1]].dropna().sort_values(A1)
    fig, ax = plt.subplots(figsize=(8, 0.34 * len(d) + 1.4))
    for i, (local, r) in enumerate(d.iterrows()):
        cor = COR_REF if local in REFS else COR_A1
        ax.plot([r[A0], r[A1]], [i, i], color=cor, lw=2, alpha=.35)
        ax.scatter(r[A0], i, s=45, color='white', edgecolor=COR_A0, lw=2, zorder=2)
        ax.scatter(r[A1], i, s=60, color=cor, edgecolor='white', lw=1.2, zorder=3)
        ax.annotate(f' {br(r[A1], casas)}{sufixo}', (max(r[A0], r[A1]), i), va='center', fontsize=8.5, xytext=(5, 0), textcoords='offset points')
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index); ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', color='#e6e6e6'); ax.set_axisbelow(True); ax.spines['left'].set_visible(False); ax.margins(x=.12)
    ax.scatter([], [], s=45, color='white', edgecolor=COR_A0, lw=2, label=str(A0)); ax.scatter([], [], s=60, color=COR_A1, label=str(A1))
    ax.legend(loc='lower right', frameon=False)
    ax.set_title(f'{indicador}, {A0} e {A1}', loc='left', weight='bold')
    ax.annotate('Fonte: INEP, Censo Escolar', (0, -0.02), xycoords='figure fraction', fontsize=7, color='#777')
    plt.tight_layout(); plt.savefig(f'graficos/{arquivo}.png', dpi=200, bbox_inches='tight'); plt.show()

def barras_var(indicador, arquivo):
    A1 = ULT[indicador]
    d = TABELA.loc[indicador, f'Variação % desde {A0}'].reindex(MUNIS + REFS).dropna().sort_values()
    fig, ax = plt.subplots(figsize=(8, 0.34 * len(d) + 1.4))
    ax.barh(d.index, d.values, color=[COR_REF if l in REFS else COR_VIZ for l in d.index])
    for i, v in enumerate(d.values):
        ax.annotate(f' {br(v)}%', (v, i), va='center', ha='left' if v >= 0 else 'right', fontsize=8.5)
    ax.axvline(0, color='#333', lw=.8); ax.margins(x=.15); ax.tick_params(axis='y', length=0)
    ax.set_title(f'{indicador}: variação de {A0} a {A1} (%)', loc='left', weight='bold')
    plt.tight_layout(); plt.savefig(f'graficos/{arquivo}.png', dpi=200, bbox_inches='tight'); plt.show()

barras_var('Matrículas: Educação básica (total)', '01_matriculas_variacao')
halteres('Escolas em atividade', '02_escolas', casas=0, refs=False)
halteres('Escolas rurais', '03_escolas_rurais', casas=0, refs=False)
barras_var('Matrículas: Creche', '04_creche_variacao')
barras_var('Matrículas: Ensino médio', '05_ensino_medio_variacao')
halteres('% das matrículas na rede municipal', '06_rede_municipal', '%')
halteres('Alunos por turma', '07_alunos_por_turma')
halteres('% de escolas com: Internet', '08_internet', '%')

# Evolução por etapa nos três recortes
etapas = ['Matrículas: Creche', 'Matrículas: Pré-escola', 'Matrículas: Fundamental, anos iniciais',
          'Matrículas: Fundamental, anos finais', 'Matrículas: Ensino médio', 'Matrículas: EJA (jovens e adultos)']
fig, axs = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for ax, rec in zip(axs, [CAND, NC, MICRO]):
    base = RECORTES.loc[etapas, (A0, rec)]
    for e in etapas:
        anos = [a for a in ANOS if a <= ULT[e]]
        serie = [100 * RECORTES.loc[e, (a, rec)] / base[e] for a in anos]
        ax.plot(anos, serie, marker='o', label=e.replace('Matrículas: ', ''))
    ax.axhline(100, color='#999', lw=.8, ls='--'); ax.set_title(rec, loc='left', weight='bold', fontsize=10)
    ax.set_xticks([a for a in ANOS if a <= ULT[etapas[0]]])
axs[0].set_ylabel(f'Matrículas ({A0} = 100)'); axs[-1].legend(fontsize=8, frameon=False, loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout(); plt.savefig('graficos/09_etapas_recortes.png', dpi=200, bbox_inches='tight'); plt.show()

## 5. Salvar e baixar (mande o zip de volta)

In [ ]:
leia = pd.DataFrame({'Item': ['Fonte', 'Anos', 'Municípios', 'Escolas', 'Professores', 'Infraestrutura', 'Recortes', 'Rede pública e variação'], 'Descrição': [
    'INEP, microdados do Censo Escolar da Educação Básica (download.inep.gov.br/dados_abertos), arquivo de escolas.',
    ', '.join(map(str, ANOS)) + '. Data de referência do Censo Escolar: última quarta-feira de maio de cada ano.',
    '; '.join(f'{n} ({c})' for c, (n, _) in sorted(MUNICIPIOS.items(), key=lambda x: x[1][0])),
    'Só escolas em atividade (TP_SITUACAO_FUNCIONAMENTO = 1). Rural = TP_LOCALIZACAO 2.',
    'QT_DOC_BAS somado por escola: quem dá aula em duas escolas conta duas vezes. Serve para comparar, não é o número de pessoas.',
    '% sobre as escolas em atividade que responderam o item. Itens que não existiam em algum ano ficam vazios.',
    f'{CAND}, {NC} e {MICRO} se sobrepõem; não some os recortes.',
    'Pública = federal + estadual + municipal. Variação: do primeiro ano até o último ano com dado do indicador (coluna "Último ano com dado").']})
with pd.ExcelWriter('censo_escolar_regioes.xlsx') as xls:
    leia.to_excel(xls, sheet_name='Leia-me', index=False)
    TABELA.round(2).to_excel(xls, sheet_name='Municípios (tabela)')
    for ano in ANOS:
        IND[ano].T.round(2).to_excel(xls, sheet_name=f'Indicadores {ano}')
    RECORTES.round(2).to_excel(xls, sheet_name='Recortes')
    LONGO.round(4).to_excel(xls, sheet_name='Dados longos', index=False)
ESCOLAS.assign(municipio=ESCOLAS['CO_MUNICIPIO'].map(NOME)).to_csv('escolas_21_municipios.csv', index=False, encoding='utf-8')
with zipfile.ZipFile('censo_escolar_regioes.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for arq in ['censo_escolar_regioes.xlsx', 'escolas_21_municipios.csv'] + [f'graficos/{g}' for g in sorted(os.listdir('graficos'))]:
        z.write(arq)
print('Salvo: censo_escolar_regioes.zip')
try:
    from google.colab import files
    files.download('censo_escolar_regioes.zip')
except ImportError:
    pass